In [9]:
import os
import operator
import asyncio

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from google import genai

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig

from typing_extensions import TypedDict
from typing import List, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [14]:
load_dotenv()

client=genai.Client(
    http_options={"api_version": "v1beta"}
)

In [3]:
class AgentState(TypedDict):
    query: str
    route: str
    confidence: float
    response: str
    probabilities: dict[str, float]

In [4]:
def billing_node(state: AgentState):
    return {
        "response": "Billing agent is handling the query"
    }

In [5]:
def technical_node(state: AgentState):
    return {
        "response": "Technical agent is handling the query"
    }

In [6]:
def shipping_node(state: AgentState):
    return {
        "response": "Shipping agent is handling the query"
    }

In [7]:
def human_node(state: AgentState):
    return {
        "response": "Human agent is handling the query"
    }

In [46]:
def router_node(state: AgentState):
    decision=client.decisions.create(
        model="gemini-decision-1",
        input=state["query"],
        questions={
            "support_route": {
                "type": "choice",
                "instructions": (
                    "Which support team should handle this customer request?"
                    "Choose the most appropriate option from the following choices"
                ),
                "criteria": {
                    "billing": "For payments, refunds, billing, invoices and account issues.",
                    "technical": "For bugs, product issues, feature requests, technical support, authentication and login or platform issues.",
                    "shipping": "For shipping, tracking, delivery, returns, logistics or order issues.",
                    "other": "For complex issues that require human intervention or escalation or if anything is unclear."
                }
            }
        }
    )

    answer=decision.answers["support_route"]

    if answer.type != "choice":
        return {
            "route": "human",
            "confidence": 0.0,
            "probabilities": {}
        }
    
    probabilities=answer.probabilities

    route=answer.choice

    if answer.confidence<0.75 or route=="other":
        route="human"
    
    return {
        "route": route,
        "confidence": answer.confidence,
        "probabilities": probabilities
    }

In [47]:
def choose_next_node(state: AgentState):
    return state["route"]

In [48]:
builder=StateGraph(AgentState)

builder.add_node("router", router_node)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_node("shipping", shipping_node)
builder.add_node("human", human_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    choose_next_node, # function that will choose the next node based on decisions api
    {
        "billing": "billing",
        "technical": "technical",
        "shipping": "shipping",
        "human": "human"
    }
)

for node in ["billing", "technical", "shipping", "human"]:
    builder.add_edge(node, END)

graph=builder.compile()

In [49]:
initial_state={
    "query": "I need help with my billing"
}

graph.invoke(initial_state)

{'query': 'I need help with my billing',
 'route': 'billing',
 'confidence': 0.9766821473021471,
 'response': 'Billing agent is handling the query',
 'probabilities': {'billing': 0.9766821473021471,
  'shipping': 0.003097805394958316,
  'technical': 0.0036938093847910364,
  'other': 0.016526237918103547}}

In [50]:
graph.invoke({
    "query": "I am not able to login into my account"
})

{'query': 'I am not able to login into my account',
 'route': 'technical',
 'confidence': 0.912149894497999,
 'response': 'Technical agent is handling the query',
 'probabilities': {'billing': 0.05802278341926879,
  'other': 0.0255247297915903,
  'technical': 0.912149894497999,
  'shipping': 0.004302592291141822}}

In [51]:
graph.invoke({
    "query": "I am not able to connect to the delivery agent"
})

{'query': 'I am not able to connect to the delivery agent',
 'route': 'shipping',
 'confidence': 0.9237445714457724,
 'response': 'Shipping agent is handling the query',
 'probabilities': {'shipping': 0.9237445714457724,
  'other': 0.054089824964663476,
  'billing': 0.002233451846726389,
  'technical': 0.01993215174283772}}